# Gated Bayesian Regime Submission

Self-contained Kaggle notebook for the conservative gated Bayesian stratigraphic-regime model. It trains a Huber signed-delta mean, residual Hilbert/state experts, and a Student-t MAP gate, then writes `/kaggle/working/submission.csv`.

## 1. Setup

In [1]:

"Imports, constants, and robust data discovery."

from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor, Ridge
from sklearn.metrics import pairwise_distances
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler

KAGGLE_ROOT = Path("/kaggle")
KAGGLE_INPUT_ROOT = Path("/kaggle/input")
KAGGLE_OUTPUT = Path("/kaggle/working")

MODEL_NAME = "gated_bayesian_regime_conservative"
MAX_TARGET_ROWS_PER_WELL = 120
RANDOM_STATE = 20260630
ANALOG_LAGS = np.arange(-40.0, 42.5, 2.5)

NU = 3.0
HSGP_M = 8
HSGP_ELL = 3.0
HSGP_PRIOR_SCALE = 0.55
STATE_RIDGE_ALPHA = 120.0
MAX_OPT_ITER = 650
GATE_MAX_ITER = 450
GATE_PRIOR_SCALE = 0.25
GATE_SHRINK_PRIOR = 0.25
HUBER_ALPHA = 0.0001
HUBER_EPSILON = 1.35

DESCRIPTOR_COLUMNS = [
    "known_rows", "known_tvt_span", "known_md_span", "known_gr_mean_last300", "known_gr_std_last300", "known_gr_slope_last300",
    "xy_span_known", "z_drop_known", "typewell_corr_last300", "typewell_mae_last300", "typewell_best_lag_corr",
    "typewell_best_lag_tvt", "typewell_best_lag_mae", "selected_analog_distance", "extrapolation_stress_score",
    "analog_stress_score", "combined_stress_score", "descriptor_cluster",
]
SMOOTH_FEATURES = [
    "distance_from_boundary_md", "eval_progress", "MD", "Z", "md_norm", "dX_from_boundary", "dY_from_boundary", "dZ_from_boundary",
    "horizontal_dist_from_boundary", "tvt_slope_last_10", "tvt_slope_last_25", "tvt_slope_last_50", "tvt_slope_last_100",
    "z_slope_last_25", "z_slope_last_50", "z_slope_last_100", "tvt_slope_last_10_projected_delta",
    "tvt_slope_last_25_projected_delta", "tvt_slope_last_50_projected_delta", "tvt_slope_last_100_projected_delta",
]
LINEAR_FEATURES = ["last_known_tvt"]
STATE_FEATURES = [
    "state_known_tvt_last", "state_known_tvt_slope", "state_known_gr_coverage", "state_known_gr_last", "state_known_gr_iqr",
    "state_known_xy_span", "state_known_z_drop", "state_known_md_span",
]
TYPEWELL_GATE_COLUMNS = [
    "typewell_corr_last300", "typewell_mae_last300", "typewell_best_lag_corr", "typewell_best_lag_mae",
    "selected_analog_distance", "extrapolation_stress_score", "analog_stress_score", "combined_stress_score",
]
FEATURE_COLUMNS = list(dict.fromkeys(SMOOTH_FEATURES + LINEAR_FEATURES))
GATE_FEATURE_COLUMNS = list(dict.fromkeys(STATE_FEATURES + TYPEWELL_GATE_COLUMNS + ["distance_log1p", "eval_progress", "abs_projected_delta_10"]))
HSGP_COORDINATES = ["distance_log1p", "eval_progress", "abs_projected_delta_10"]


def find_competition_base():
    if KAGGLE_INPUT_ROOT.exists():
        candidates = []
        for path in KAGGLE_INPUT_ROOT.rglob("sample_submission.csv"):
            base = path.parent
            if (base / "train").exists() and (base / "test").exists():
                candidates.append(base)
        if candidates:
            return sorted(candidates, key=lambda item: len(str(item)))[0]
    for root in [Path.cwd(), *Path.cwd().parents]:
        base = root / "data" / "raw"
        if (base / "train").exists() and (base / "test").exists() and (base / "sample_submission.csv").exists():
            return base
    return Path("data/raw")


BASE_DIR = find_competition_base()
TRAIN_DIR = BASE_DIR / "train"
TEST_DIR = BASE_DIR / "test"
SAMPLE_PATH = BASE_DIR / "sample_submission.csv"
OUTPUT_PATH = KAGGLE_OUTPUT / "submission.csv" if KAGGLE_ROOT.exists() else Path("artifacts/reports/submission__gated_bayesian_regime__2026-06-30.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_PATH:", OUTPUT_PATH)


BASE_DIR: C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\data\raw
OUTPUT_PATH: artifacts\reports\submission__gated_bayesian_regime__2026-06-30.csv


## 2. Helpers

In [2]:

"Loaders, descriptors, slope-core state features, residual experts, and the Bayesian gate."

def discover_wells(directory):
    return sorted(path.name.replace("__horizontal_well.csv", "") for path in Path(directory).glob("*__horizontal_well.csv"))


def load_horizontal(path):
    df = pd.read_csv(path)
    df.index.name = "row_idx"
    df["row_idx"] = df.index.astype(int)
    return df


def load_typewell(directory, wellname):
    path = Path(directory) / f"{wellname}__typewell.csv"
    if not path.exists():
        return pd.DataFrame(columns=["TVT", "GR", "Geology"])
    return pd.read_csv(path)


def load_sample_submission(path):
    sample = pd.read_csv(path)
    split = sample["id"].str.rsplit("_", n=1, expand=True)
    sample["wellname"] = split[0]
    sample["row_idx"] = split[1].astype(int)
    return sample


def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce").astype(float)


def deterministic_sample_positions(mask, max_rows):
    positions = np.flatnonzero(mask.to_numpy())
    if len(positions) <= max_rows:
        return positions
    return positions[np.linspace(0, len(positions) - 1, max_rows).astype(int)]


def window_slope(md, values, end_pos, window):
    start_pos = max(0, end_pos - window + 1)
    x0, x1 = float(md.iloc[start_pos]), float(md.iloc[end_pos])
    y0, y1 = float(values.iloc[start_pos]), float(values.iloc[end_pos])
    if not np.isfinite([x0, x1, y0, y1]).all() or abs(x1 - x0) < 1e-9:
        return np.nan
    return (y1 - y0) / (x1 - x0)


def safe_corr(left, right):
    a = np.asarray(left, dtype=float)
    b = np.asarray(right, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    if int(mask.sum()) < 5 or float(np.nanstd(a[mask])) < 1e-9 or float(np.nanstd(b[mask])) < 1e-9:
        return np.nan
    return float(np.corrcoef(a[mask], b[mask])[0, 1])


def linear_slope(x_values, y_values):
    x = np.asarray(x_values, dtype=float)
    y = np.asarray(y_values, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if int(mask.sum()) < 5:
        return np.nan
    return float(np.polyfit(x[mask], y[mask], 1)[0])


def interp_typewell_gr(typewell, tvt_values, lag=0.0):
    if typewell.empty or not {"TVT", "GR"}.issubset(typewell.columns):
        return np.full(len(tvt_values), np.nan)
    clean = typewell[["TVT", "GR"]].dropna().sort_values("TVT")
    if len(clean) < 5:
        return np.full(len(tvt_values), np.nan)
    return np.interp(np.asarray(tvt_values, dtype=float) + lag, clean["TVT"].to_numpy(), clean["GR"].to_numpy(), left=np.nan, right=np.nan)


def build_descriptor(wellname, directory):
    horizontal = load_horizontal(Path(directory) / f"{wellname}__horizontal_well.csv")
    typewell = load_typewell(directory, wellname)
    known = horizontal.loc[horizontal["TVT_input"].notna()].copy()
    if known.empty:
        known = horizontal.head(1).copy()
    gr = known["GR"].astype(float).clip(0, 300)
    last300 = known.tail(300).copy()
    gr_last300 = last300["GR"].astype(float).clip(0, 300)
    known_tvt = known["TVT_input"].astype(float) if "TVT_input" in known else known.get("TVT", pd.Series(dtype=float)).astype(float)
    last_known_tvt = float(known_tvt.dropna().iloc[-1]) if known_tvt.notna().any() else np.nan
    desc = {
        "wellname": wellname,
        "known_rows": float(len(known)),
        "known_tvt_span": float(np.nanmax(known_tvt) - np.nanmin(known_tvt)) if known_tvt.notna().sum() else np.nan,
        "known_md_span": float(known["MD"].max() - known["MD"].min()) if "MD" in known else np.nan,
        "known_gr_mean_last300": float(gr_last300.mean()) if gr_last300.notna().any() else np.nan,
        "known_gr_std_last300": float(gr_last300.std(ddof=0)) if gr_last300.notna().any() else np.nan,
        "known_gr_slope_last300": linear_slope(last300["MD"], gr_last300) if len(last300) else np.nan,
        "xy_span_known": float(np.sqrt((known["X"].iloc[-1] - known["X"].iloc[0]) ** 2 + (known["Y"].iloc[-1] - known["Y"].iloc[0]) ** 2)) if {"X", "Y"}.issubset(known.columns) and len(known) else np.nan,
        "z_drop_known": float(known["Z"].iloc[-1] - known["Z"].iloc[0]) if "Z" in known and len(known) else np.nan,
        "last_known_tvt": last_known_tvt,
    }
    if len(last300) and np.isfinite(last_known_tvt):
        type_gr = interp_typewell_gr(typewell, last300["TVT_input"].astype(float), lag=0.0)
        desc["typewell_corr_last300"] = safe_corr(gr_last300, type_gr)
        desc["typewell_mae_last300"] = float(np.nanmean(np.abs(gr_last300.to_numpy(float) - type_gr))) if np.isfinite(type_gr).any() else np.nan
        best = {"corr": -np.inf, "lag": np.nan, "mae": np.nan}
        for lag in ANALOG_LAGS:
            shifted = interp_typewell_gr(typewell, last300["TVT_input"].astype(float), lag=lag)
            corr = safe_corr(gr_last300, shifted)
            if np.isfinite(corr) and corr > best["corr"]:
                mae = float(np.nanmean(np.abs(gr_last300.to_numpy(float) - shifted))) if np.isfinite(shifted).any() else np.nan
                best = {"corr": corr, "lag": float(lag), "mae": mae}
        desc["typewell_best_lag_corr"] = best["corr"] if np.isfinite(best["corr"]) else np.nan
        desc["typewell_best_lag_tvt"] = best["lag"]
        desc["typewell_best_lag_mae"] = best["mae"]
    else:
        desc["typewell_corr_last300"] = np.nan
        desc["typewell_mae_last300"] = np.nan
        desc["typewell_best_lag_corr"] = np.nan
        desc["typewell_best_lag_tvt"] = np.nan
        desc["typewell_best_lag_mae"] = np.nan
    desc["extrapolation_stress_score"] = np.nan
    desc["analog_stress_score"] = np.nan
    desc["combined_stress_score"] = np.nan
    desc["descriptor_cluster"] = np.nan
    return desc


def add_nearest_analog_distance(train_desc, test_desc=None):
    train = train_desc.copy().reset_index(drop=True)
    test = None if test_desc is None else test_desc.copy().reset_index(drop=True)
    cols = [col for col in DESCRIPTOR_COLUMNS if col not in {"selected_analog_distance", "extrapolation_stress_score", "analog_stress_score", "combined_stress_score", "descriptor_cluster"} and col in train.columns]
    usable = []
    for col in cols:
        train[col] = pd.to_numeric(train[col], errors="coerce")
        if test is not None and col in test.columns:
            test[col] = pd.to_numeric(test[col], errors="coerce")
        if train[col].notna().any():
            usable.append(col)
    pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
    train_x = pipe.fit_transform(train[usable])
    train_dist = pairwise_distances(train_x, train_x)
    np.fill_diagonal(train_dist, np.inf)
    train["selected_analog_distance"] = np.nanmin(train_dist, axis=1)
    train["analog_stress_score"] = train["selected_analog_distance"]
    train["extrapolation_stress_score"] = train["known_md_span"]
    train["combined_stress_score"] = train["selected_analog_distance"]
    if test is not None and not test.empty:
        for col in usable:
            if col not in test.columns:
                test[col] = np.nan
        test_x = pipe.transform(test[usable])
        dist = pairwise_distances(test_x, train_x)
        test["selected_analog_distance"] = np.nanmin(dist, axis=1)
        test["analog_stress_score"] = test["selected_analog_distance"]
        test["extrapolation_stress_score"] = test["known_md_span"]
        test["combined_stress_score"] = test["selected_analog_distance"]
        return train, test
    return train, test


def build_feature_rows(wellname, directory, descriptor_row, row_indices=None, max_target_rows=None, require_target=True):
    df = load_horizontal(Path(directory) / f"{wellname}__horizontal_well.csv").sort_values("MD").reset_index(drop=True)
    df["row_idx"] = df.index.astype(int)
    known_mask = df["TVT_input"].notna()
    if int(known_mask.sum()) == 0:
        return pd.DataFrame()
    last_known_pos = int(np.flatnonzero(known_mask.to_numpy())[-1])
    tvt_input = safe_numeric(df["TVT_input"])
    md = safe_numeric(df["MD"])
    x = safe_numeric(df["X"])
    y = safe_numeric(df["Y"])
    z = safe_numeric(df["Z"])
    gr = safe_numeric(df["GR"]).clip(0, 300)
    if row_indices is not None:
        target_positions = np.flatnonzero(df["row_idx"].isin([int(value) for value in row_indices]).to_numpy())
    else:
        target_mask = df["TVT_input"].isna()
        if require_target and "TVT" in df.columns:
            target_mask = target_mask & df["TVT"].notna()
        target_positions = deterministic_sample_positions(target_mask, max_target_rows) if max_target_rows is not None else np.flatnonzero(target_mask.to_numpy())
    if len(target_positions) == 0:
        return pd.DataFrame()
    all_target_positions = target_positions if row_indices is not None else np.flatnonzero(df["TVT_input"].isna().to_numpy())
    eval_rank = {int(pos): rank for rank, pos in enumerate(all_target_positions)}
    eval_denominator = max(len(all_target_positions) - 1, 1)
    known_gr = gr.iloc[:last_known_pos + 1]
    known_gr_finite = known_gr.dropna()
    gr_fill = float(known_gr.tail(300).median()) if known_gr.notna().any() else 0.0
    slope_windows = [10, 25, 50, 100]
    tvt_slopes = {f"tvt_slope_last_{window}": window_slope(md, tvt_input, last_known_pos, window) for window in slope_windows}
    z_slopes = {f"z_slope_last_{window}": window_slope(md, z, last_known_pos, window) for window in slope_windows}
    horizontal_dist = np.sqrt(x.diff().fillna(0.0) ** 2 + y.diff().fillna(0.0) ** 2).cumsum()
    last_known_tvt = float(tvt_input.iloc[last_known_pos])
    known_md_span = float(md.iloc[last_known_pos] - md.iloc[0]) if last_known_pos > 0 else 0.0
    known_tvt_delta = float(tvt_input.iloc[last_known_pos] - tvt_input.iloc[0]) if last_known_pos > 0 else 0.0
    rows = []
    for pos in target_positions:
        distance_md = float(md.iloc[pos] - md.iloc[last_known_pos])
        tvt_slope_10 = tvt_slopes["tvt_slope_last_10"]
        projected_delta_10 = tvt_slope_10 * distance_md if np.isfinite(tvt_slope_10) else np.nan
        row = {
            "wellname": wellname,
            "row_idx": int(df["row_idx"].iloc[pos]),
            "TVT": float(df["TVT"].iloc[pos]) if "TVT" in df.columns and pd.notna(df["TVT"].iloc[pos]) else np.nan,
            "last_known_tvt": last_known_tvt,
            "MD": float(md.iloc[pos]),
            "Z": float(z.iloc[pos]),
            "GR": float(gr.iloc[pos]) if np.isfinite(gr.iloc[pos]) else gr_fill,
            "delta_tvt_from_boundary": float(df["TVT"].iloc[pos]) - last_known_tvt if "TVT" in df.columns and pd.notna(df["TVT"].iloc[pos]) else np.nan,
            "md_norm": float((md.iloc[pos] - md.min()) / (md.max() - md.min() + 1e-9)),
            "distance_from_boundary_md": distance_md,
            "distance_log1p": float(np.log1p(max(distance_md, 0.0))),
            "eval_progress": float(eval_rank.get(int(pos), 0) / eval_denominator),
            "dX_from_boundary": float(x.iloc[pos] - x.iloc[last_known_pos]),
            "dY_from_boundary": float(y.iloc[pos] - y.iloc[last_known_pos]),
            "dZ_from_boundary": float(z.iloc[pos] - z.iloc[last_known_pos]),
            "horizontal_dist_from_boundary": float(horizontal_dist.iloc[pos] - horizontal_dist.iloc[last_known_pos]),
            "abs_projected_delta_10": float(abs(projected_delta_10)) if np.isfinite(projected_delta_10) else np.nan,
            "state_known_tvt_last": last_known_tvt,
            "state_known_tvt_slope": known_tvt_delta / max(abs(known_md_span), 1.0),
            "state_known_gr_coverage": float(known_gr.notna().mean()) if len(known_gr) else 0.0,
            "state_known_gr_last": float(known_gr_finite.iloc[-1]) if len(known_gr_finite) else gr_fill,
            "state_known_gr_iqr": float(known_gr_finite.quantile(0.75) - known_gr_finite.quantile(0.25)) if len(known_gr_finite) >= 4 else 0.0,
            "state_known_xy_span": float(np.hypot(x.iloc[last_known_pos] - x.iloc[0], y.iloc[last_known_pos] - y.iloc[0])) if last_known_pos > 0 else 0.0,
            "state_known_z_drop": float(z.iloc[last_known_pos] - z.iloc[0]) if last_known_pos > 0 else 0.0,
            "state_known_md_span": known_md_span,
        }
        for name, value in tvt_slopes.items():
            row[name] = float(value) if np.isfinite(value) else np.nan
            row[f"{name}_projected_delta"] = row[name] * distance_md if np.isfinite(row[name]) else np.nan
        for name, value in z_slopes.items():
            row[name] = float(value) if np.isfinite(value) else np.nan
        for col in DESCRIPTOR_COLUMNS:
            row[col] = float(descriptor_row[col]) if col in descriptor_row and pd.notna(descriptor_row[col]) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def softplus(value):
    return np.logaddexp(0.0, value)


def sigmoid(value):
    return 1.0 / (1.0 + np.exp(-np.clip(value, -35, 35)))


def softmax(logits):
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(shifted)
    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)


def fit_coord_preprocessor(frame, columns):
    medians = frame[columns].median(numeric_only=True)
    filled = frame[columns].fillna(medians)
    means = filled.mean()
    stds = filled.std().replace(0.0, 1.0).fillna(1.0)
    return medians, means, stds


def apply_coord_preprocessor(frame, columns, medians, means, stds):
    filled = frame[columns].fillna(medians)
    return ((filled - means) / stds).to_numpy(float)


def hilbert_basis_1d(values):
    clipped = np.clip(values, -HSGP_ELL, HSGP_ELL)
    shifted = (clipped + HSGP_ELL) / (2.0 * HSGP_ELL)
    return np.column_stack([np.sqrt(1.0 / HSGP_ELL) * np.sin(np.pi * j * shifted) for j in range(1, HSGP_M + 1)])


def additive_hilbert_design(frame, model):
    coords = apply_coord_preprocessor(frame, HSGP_COORDINATES, model["medians"], model["means"], model["stds"])
    return np.column_stack([hilbert_basis_1d(coords[:, idx]) for idx in range(coords.shape[1])])


def residual_student_t_objective(params, X, y):
    intercept = params[0]
    beta = params[1:-1]
    sigma = softplus(params[-1]) + 1e-6
    residual = y - (intercept + X @ beta)
    scaled = residual / sigma
    nll = np.sum(np.log(sigma) + 0.5 * (NU + 1.0) * np.log1p((scaled ** 2) / NU))
    return float(nll + 0.5 * np.sum((beta / HSGP_PRIOR_SCALE) ** 2) + 0.5 * (intercept / 1.0) ** 2 + 0.5 * (sigma / 1.5) ** 2)


def fit_hsgp_residual_model(train_frame, residual):
    medians, means, stds = fit_coord_preprocessor(train_frame, HSGP_COORDINATES)
    model = {"medians": medians, "means": means, "stds": stds}
    X = additive_hilbert_design(train_frame, model)
    y_mean = float(np.mean(residual))
    y_std = float(np.std(residual) + 1e-9)
    y = (residual - y_mean) / y_std
    start = np.zeros(X.shape[1] + 2, dtype=float)
    start[-1] = np.log(np.expm1(1.0))
    result = minimize(residual_student_t_objective, start, args=(X, y), method="L-BFGS-B", options={"maxiter": MAX_OPT_ITER, "ftol": 1e-8, "maxls": 50})
    model.update({"params": result.x, "y_mean": y_mean, "y_std": y_std})
    return model


def predict_hsgp_residual(model, frame):
    X = additive_hilbert_design(frame, model)
    params = model["params"]
    return (params[0] + X @ params[1:-1]) * model["y_std"] + model["y_mean"]


def gate_objective(params, X, component_preds, residual):
    n_features = X.shape[1]
    n_components = component_preds.shape[1]
    size_w = n_features * n_components
    W = params[:size_w].reshape(n_features, n_components)
    b = params[size_w:size_w + n_components]
    shrink_raw = params[size_w + n_components]
    sigma = softplus(params[-1]) + 1e-6
    weights = softmax(X @ W + b)
    pred = sigmoid(shrink_raw) * np.sum(weights * component_preds, axis=1)
    error = residual - pred
    scaled = error / sigma
    nll = np.sum(np.log(sigma) + 0.5 * (NU + 1.0) * np.log1p((scaled ** 2) / NU))
    shrink_prior_logit = np.log(GATE_SHRINK_PRIOR / (1.0 - GATE_SHRINK_PRIOR))
    return float(nll + 0.5 * np.sum((W / GATE_PRIOR_SCALE) ** 2) + 0.5 * np.sum((b / 1.0) ** 2) + 0.5 * ((shrink_raw - shrink_prior_logit) / 0.75) ** 2 + 0.5 * (sigma / 1.5) ** 2)


def fit_gate_model(X_train, component_train, residual):
    n_features = X_train.shape[1]
    n_components = component_train.shape[1]
    start = np.zeros(n_features * n_components + n_components + 2, dtype=float)
    start[-2] = np.log(GATE_SHRINK_PRIOR / (1.0 - GATE_SHRINK_PRIOR))
    start[-1] = np.log(np.expm1(np.std(residual) + 1e-6))
    result = minimize(gate_objective, start, args=(X_train, component_train, residual), method="L-BFGS-B", options={"maxiter": GATE_MAX_ITER, "ftol": 1e-8, "maxls": 50})
    return {"params": result.x, "n_features": n_features, "n_components": n_components}


def predict_gate_model(model, X, component_preds):
    params = model["params"]
    n_features = model["n_features"]
    n_components = model["n_components"]
    size_w = n_features * n_components
    W = params[:size_w].reshape(n_features, n_components)
    b = params[size_w:size_w + n_components]
    weights = softmax(X @ W + b)
    shrink = sigmoid(params[size_w + n_components])
    return shrink * np.sum(weights * component_preds, axis=1), weights, shrink


## 3. Train Gated Regime

In [3]:

"Build training rows and fit the Huber mean, residual experts, and conservative gate."

train_wells = discover_wells(TRAIN_DIR)
test_wells = discover_wells(TEST_DIR)
train_desc = pd.DataFrame([build_descriptor(well, TRAIN_DIR) for well in train_wells])
test_desc = pd.DataFrame([build_descriptor(well, TEST_DIR) for well in test_wells])
train_desc, test_desc = add_nearest_analog_distance(train_desc, test_desc)

train_frames = []
for row in train_desc.to_dict(orient="records"):
    frame = build_feature_rows(row["wellname"], TRAIN_DIR, row, max_target_rows=MAX_TARGET_ROWS_PER_WELL, require_target=True)
    if not frame.empty:
        train_frames.append(frame)
train_df = pd.concat(train_frames, ignore_index=True)
train_df[GATE_FEATURE_COLUMNS] = train_df[GATE_FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
train_df[FEATURE_COLUMNS] = train_df[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)

y_delta = train_df["delta_tvt_from_boundary"].to_numpy(float)
huber_model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler()), ("model", HuberRegressor(alpha=HUBER_ALPHA, epsilon=HUBER_EPSILON, max_iter=1200))])
huber_model.fit(train_df[FEATURE_COLUMNS], y_delta)
huber_train_delta = huber_model.predict(train_df[FEATURE_COLUMNS])
train_residual = y_delta - huber_train_delta

hsgp_model = fit_hsgp_residual_model(train_df, train_residual)
hsgp_train = predict_hsgp_residual(hsgp_model, train_df)
state_model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler()), ("ridge", Ridge(alpha=STATE_RIDGE_ALPHA))])
state_model.fit(train_df[GATE_FEATURE_COLUMNS], train_residual)
state_train = state_model.predict(train_df[GATE_FEATURE_COLUMNS])

gate_imputer = SimpleImputer(strategy="median")
gate_scaler = StandardScaler()
X_gate_train = gate_scaler.fit_transform(gate_imputer.fit_transform(train_df[GATE_FEATURE_COLUMNS]))
component_train = np.column_stack([np.zeros(len(train_df)), hsgp_train, state_train, 0.5 * hsgp_train + 0.5 * state_train])
gate_model = fit_gate_model(X_gate_train, component_train, train_residual)
gate_correction, gate_weights, gate_shrink = predict_gate_model(gate_model, X_gate_train, component_train)

train_pred = train_df["last_known_tvt"].to_numpy(float) + huber_train_delta + gate_correction
train_rmse = float(np.sqrt(np.mean((train_df["TVT"].to_numpy(float) - train_pred) ** 2)))
print("model:", MODEL_NAME)
print("train_wells:", len(train_wells))
print("test_wells:", len(test_wells))
print("train_rows:", len(train_df))
print("train_rmse_in_sample:", train_rmse)
print("learned_shrink:", float(gate_shrink))
print("mean_gate_weights:", gate_weights.mean(axis=0).tolist())


model: gated_bayesian_regime_conservative
train_wells: 773
test_wells: 3
train_rows: 92760
train_rmse_in_sample: 13.789964609853257
learned_shrink: 0.990623848388115
mean_gate_weights: [0.22657365837249954, 0.211910307094622, 0.490169778472647, 0.07134625606023884]


## 4. Write Submission

In [4]:

"Generate predictions in sample_submission order and write submission.csv."

sample = load_sample_submission(SAMPLE_PATH)
predictions = {}
test_desc_map = {row["wellname"]: row for row in test_desc.to_dict(orient="records")}

for wellname, group in sample.groupby("wellname", sort=True):
    descriptor_row = test_desc_map.get(wellname)
    if descriptor_row is None:
        descriptor_row = build_descriptor(wellname, TEST_DIR)
        descriptor_row["selected_analog_distance"] = float(np.nanmedian(train_desc["selected_analog_distance"]))
    frame = build_feature_rows(wellname, TEST_DIR, descriptor_row, row_indices=group["row_idx"].tolist(), max_target_rows=None, require_target=False)
    if frame.empty:
        continue
    frame[GATE_FEATURE_COLUMNS] = frame[GATE_FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
    frame[FEATURE_COLUMNS] = frame[FEATURE_COLUMNS].replace([np.inf, -np.inf], np.nan)
    mean_delta = huber_model.predict(frame[FEATURE_COLUMNS])
    hsgp_pred = predict_hsgp_residual(hsgp_model, frame)
    state_pred = state_model.predict(frame[GATE_FEATURE_COLUMNS])
    X_gate = gate_scaler.transform(gate_imputer.transform(frame[GATE_FEATURE_COLUMNS]))
    components = np.column_stack([np.zeros(len(frame)), hsgp_pred, state_pred, 0.5 * hsgp_pred + 0.5 * state_pred])
    correction, weights, shrink = predict_gate_model(gate_model, X_gate, components)
    pred_tvt = frame["last_known_tvt"].to_numpy(float) + mean_delta + correction
    for row_idx, value in zip(frame["row_idx"].astype(int), pred_tvt):
        predictions[f"{wellname}_{row_idx}"] = float(value)

submission = sample[["id"]].copy()
submission["tvt"] = submission["id"].map(predictions)
assert list(submission.columns) == ["id", "tvt"]
assert len(submission) == len(sample)
assert submission["id"].equals(sample["id"])
assert submission["tvt"].notna().all()
assert np.isfinite(submission["tvt"].to_numpy()).all()

submission.to_csv(OUTPUT_PATH, index=False)
print("submission_rows:", len(submission))
print("submission_path:", OUTPUT_PATH)
print("prediction_min:", float(submission["tvt"].min()))
print("prediction_max:", float(submission["tvt"].max()))
submission.head()


submission_rows: 14151
submission_path: artifacts\reports\submission__gated_bayesian_regime__2026-06-30.csv
prediction_min: 11607.46131065447
prediction_max: 12228.180410424426


,id,tvt
0,000d7d20_1442,11744.304569
1,000d7d20_1443,11744.262985
2,000d7d20_1444,11744.235450
3,000d7d20_1445,11744.215902
4,000d7d20_1446,11744.201555
